# SONARA: Chapter 3 Pipeline (Data Preparation, CNN–BiLSTM, Evaluation, Zone-Level Attribution)

This notebook runs the whole offline pipeline in one place and follows the revision plan agreed for the panel comments.

| Notebook part | Manuscript section | Panel rows it supports |
| :- | :- | :- |
| 1. Setup and configuration | — | — |
| 2. Datasets and label consolidation | 3.1.1 | 4 |
| 3. Preprocessing: 16 kHz, mono, one 4-s crop per file | 3.1.2, Figure 3.1 | — |
| 4. Features: log-mel + deltas (3 × 64 × 400) | 3.1.2, 3.1.3, Figure 3.2 | 4 |
| 5. Splitting (70/15/15, grouped by recording) and augmentation | 3.1.2, 3.1.4 | — |
| 6. CNN–BiLSTM and comparison models | 3.2.1 | 3, 15 |
| 7. Loss, optimizer, scheduler, early stopping | 3.2.2, 3.2.3 | 7 – 10, 18 |
| 8. Training over several seeds | 3.2.3 | 15 |
| 9. Evaluation, baselines and Table 3.1 | 3.2.4, 3.3.5 | 11, 12, 15 – 17 |
| 10. External test on PVSD (if present) | 3.3.5 | 5 |
| 11. Gateway inference and zone-level attribution (simulation) | 3.3.2 – 3.3.4 | 5 |
| 12. Values to copy into the manuscript | 3.1, 3.2 | — |

**Where this notebook differs from the current manuscript text** (the text should be changed to match, not the other way round):

- **One 4-second crop per file, not 1 to 2 second windows.** SmartEars labels describe a whole 5-second clip. A 1 to 2 second slice of a "Sick" clip may contain no cough at all, so slicing would create wrong labels.
- **SmartEars only for training, validation and testing.** The Poultry Vocalization Signal Dataset (PVSD) is kept out and used as an external test set, because its 346 files come from very few groups of birds.
- **Classes are not balanced by removing data.** The real 2 : 1 ratio is kept and handled by the class weights in the loss.
- **Augmentation** is random crop position, random gain, and time and frequency masking. Noise mixing is available but off by default.
- **The decision threshold** is reported at 0.5 and also at a value chosen on the validation set to reach a target distress recall.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. With the default 4 models × 5 seeds, expect a few hours. Finished runs are saved, so rerunning continues where it stopped.

## 1. Setup and configuration

In [ ]:
import os, re, copy, glob, json, random, time
from collections import deque
from math import gcd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ---- Data location -------------------------------------------------------
REPO_URL = "https://github.com/tagalogclark123-cmyk/SONARA-GITHUB.git"
REPO_DIR = "/content/SONARA-GITHUB"
OUT_DIR = "/content/sonara_outputs"
SAVE_TO_DRIVE = False         # True = keep outputs in Google Drive so a disconnect loses nothing
DRIVE_OUT_DIR = "/content/drive/MyDrive/SONARA/sonara_outputs"

# Each dataset is a folder of class sub-folders inside the repo. The label map sends the
# original labels to three source categories. Datasets whose folder is missing are skipped.
DATASETS = [
    {"name": "SmartEars", "root": "Smartears",
     "label_map": {"Healthy": "Normal", "None": "Noise", "Sick": "Distress"}},
    {"name": "PVSD", "root": "PVSD",
     "label_map": {"Healthy": "Normal", "Noise": "Noise", "Unhealthy": "Distress"}},
]
TRAIN_DATASETS = ["SmartEars"]   # used for train / validation / test
EXTERNAL_DATASETS = ["PVSD"]     # never trained on; external test only

# ---- 3.1.4 Splitting --------------------------------------------------------
SEEDS = [7, 21, 42, 77, 99]   # each seed = one split + one training run per model
SPLIT_FOLDS = (14, 3, 3)      # 20 folds -> 70% train, 15% validation, 15% test
GROUP_BY = "recording"        # "recording" | "device_day" | "device" (kept together in one split)

# ---- 3.1.2 Preprocessing and features ---------------------------------------
SAMPLE_RATE = 16000
CROP_SEC = 4.0                # one crop per file
N_FFT = 400                   # 25 ms STFT window
HOP_LENGTH = 160              # 10 ms hop
N_MELS = 64
FMIN, FMAX = 50, 8000

# ---- 3.1.2 Augmentation (training split only) -------------------------------
AUG_GAIN_DB = (-6.0, 6.0)     # random gain on the waveform
AUG_TIME_MASK_PROB = 0.4      # chance of masking a block of time frames
AUG_FREQ_MASK_PROB = 0.4      # chance of masking a block of mel bands
AUG_NOISE_PROB = 0.0          # >0 mixes in farm noise from training-split "None" clips
AUG_SNR_DB = (5.0, 20.0)

# ---- 3.2.2 / 3.2.3 Training -------------------------------------------------
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02        # epsilon in Equation 3-3
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 8
LR_PATIENCE = 2
LR_FACTOR = 0.5
DROPOUT = 0.3

# ---- 3.3.2 Decision threshold ------------------------------------------------
DEFAULT_THRESHOLD = 0.5
TARGET_RECALL = 0.90          # validation distress recall the tuned threshold must reach

# ---- Debugging --------------------------------------------------------------
QUICK_RUN = None              # e.g. 60 = use 60 clips per class, 2 epochs, 2 seeds

CLASS_NAMES = ["Non-distress", "Distress"]
CROP_SAMPLES = int(round(CROP_SEC * SAMPLE_RATE))
NUM_WORKERS = 2 if os.name == "posix" else 0
if QUICK_RUN:
    MAX_EPOCHS, SEEDS = 2, SEEDS[:2]
PRIMARY_SEED = SEEDS[0]       # the run used for figures, the saved model and the simulation


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = DRIVE_OUT_DIR
RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)
print("Outputs:", OUT_DIR)

In [ ]:
# Download the repository (code + SmartEars audio). This takes a few minutes the first time.
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
print("Repo folder:", REPO_DIR, "| exists:", os.path.isdir(REPO_DIR))

## 2. Datasets and label consolidation (Section 3.1.1)

Original labels are consolidated into two target classes: **Non-distress** (healthy/normal and none/noise) and **Distress** (sick/unhealthy). The three-way source category is kept as a column so that Non-distress errors can be examined separately for normal and noise samples.

The classes are **not** balanced by dropping clips. The counts below are the real ones, and the imbalance is handled by the class weights in the loss (Equation 3-1).

In [ ]:
SMARTEARS_NAME = re.compile(r"^(rpi\d+)-(\d{8})(\d{6})(?:-(\d+))?$")


def build_manifest():
    rows = []
    for ds in DATASETS:
        root = os.path.join(REPO_DIR, ds["root"])
        if not os.path.isdir(root):
            print(f"[skip] {ds['name']}: folder '{ds['root']}' not found in the repo")
            continue
        for folder, source in ds["label_map"].items():
            for path in sorted(glob.glob(os.path.join(root, folder, "*.wav"))):
                stem = os.path.splitext(os.path.basename(path))[0]
                m = SMARTEARS_NAME.match(stem)
                if m:   # device id + start time identify the source recording
                    device_id, day = m.group(1), m.group(2)
                    recording = f"{device_id}-{day}{m.group(3)}"
                else:
                    device_id, day, recording = stem, stem, stem
                rows.append({
                    "path": path, "dataset": ds["name"], "original_label": folder,
                    "source": source, "label": int(source == "Distress"),
                    "recording": f"{ds['name']}:{recording}",
                    "device_day": f"{ds['name']}:{device_id}-{day}",
                    "device": f"{ds['name']}:{device_id}",
                })
    return pd.DataFrame(rows)


manifest = build_manifest()
assert len(manifest) > 0, "No audio files found. Check REPO_DIR and DATASETS."
if QUICK_RUN:
    manifest = manifest.groupby(["dataset", "source"]).head(QUICK_RUN).reset_index(drop=True)

manifest["role"] = np.where(manifest["dataset"].isin(TRAIN_DATASETS), "train/val/test",
                            np.where(manifest["dataset"].isin(EXTERNAL_DATASETS), "external test", "unused"))
print("\nClips per dataset and original label:")
print(manifest.groupby(["dataset", "role", "original_label", "source"]).size().to_string())

main_idx = manifest.index[manifest["role"] == "train/val/test"].to_numpy()
external_idx = manifest.index[manifest["role"] == "external test"].to_numpy()
counts = manifest.loc[main_idx, "label"].value_counts().sort_index()
print(f"\nTarget classes in the training datasets: Non-distress {counts.get(0, 0):,} | Distress {counts.get(1, 0):,}")

# How many clips are segments of the same source recording (this is why the split is grouped).
per_recording = manifest.loc[main_idx].groupby("recording").size()
print(f"Source recordings: {len(per_recording):,} | recordings cut into more than one clip: "
      f"{int((per_recording > 1).sum()):,} (up to {int(per_recording.max())} clips each)")

## 3. Preprocessing (Section 3.1.2)

Every recording is resampled to 16 kHz and converted to mono. Each file then gives **one 4-second crop**: a random position during training, and the centre of the clip for validation and testing.

In [ ]:
def load_audio(path, target_sr=SAMPLE_RATE):
    """Read a file, average channels to mono, and resample to target_sr."""
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != target_sr:
        g = gcd(int(sr), int(target_sr))
        wav = resample_poly(wav, target_sr // g, sr // g).astype(np.float32)
    return wav, sr


t0 = time.time()
clips = []            # one int16 waveform per file, at 16 kHz mono
for i, path in enumerate(manifest["path"]):
    wav, _ = load_audio(path)
    clips.append(np.clip(wav * 32767.0, -32768, 32767).astype(np.int16))
    if (i + 1) % 500 == 0:
        print(f"  loaded {i + 1}/{len(manifest)} clips ({time.time() - t0:.0f}s)")
manifest["seconds"] = [len(c) / SAMPLE_RATE for c in clips]
print(f"Loaded {len(clips)} clips in {time.time() - t0:.0f}s")
print(manifest.groupby("dataset")["seconds"].describe()[["count", "min", "50%", "max"]].round(2).to_string())
print(f"Files shorter than {CROP_SEC:.0f} s (repeated to fill the crop):")
print(manifest[manifest["seconds"] < CROP_SEC].groupby(["dataset", "source"]).size().to_string() or "  none")

In [ ]:
# Figure 3.1: example waveform before and after resampling to 16 kHz mono.
example_path = manifest.loc[manifest["label"] == 1, "path"].iloc[0]
raw, raw_sr = sf.read(example_path, dtype="float32", always_2d=True)
resampled, _ = load_audio(example_path)

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
axes[0].plot(np.arange(len(raw)) / raw_sr, raw[:, 0], linewidth=0.5)
axes[0].set_title(f"Original: {raw_sr} Hz, {raw.shape[1]} channel(s)")
axes[1].plot(np.arange(len(resampled)) / SAMPLE_RATE, resampled, linewidth=0.5)
axes[1].set_title(f"After preprocessing: {SAMPLE_RATE} Hz, mono")
axes[1].set_xlabel("Time (s)")
for ax in axes:
    ax.set_ylabel("Amplitude")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_3_1_resampling.png"), dpi=200)
plt.show()

In [ ]:
def crop(clip, start=None):
    """Fixed-length crop as float32 in [-1, 1]. Short clips are repeated, not zero-padded,
    so that a block of silence cannot become a shortcut for the label."""
    if len(clip) < CROP_SAMPLES:
        clip = np.tile(clip, int(np.ceil(CROP_SAMPLES / max(len(clip), 1))))
    if start is None:                                   # centre crop
        start = (len(clip) - CROP_SAMPLES) // 2
    return clip[start:start + CROP_SAMPLES].astype(np.float32) / 32768.0

## 4. Feature representation (Sections 3.1.2 and 3.1.3)

Each 4-second crop is passed through a short-time Fourier transform (25 ms window, 10 ms hop) and a 64-band mel filterbank, and the energy in each band is log-compressed. The log-mel matrix and its first and second derivatives are stacked into a **3 × 64 × 400 tensor**. Each channel is standardized per crop so that the features do not depend on microphone gain.

In [ ]:
N_FRAMES = CROP_SAMPLES // HOP_LENGTH


class LogMelDeltas(nn.Module):
    """Waveform (batch, samples) -> tensor (batch, 3, n_mels, frames)."""

    def __init__(self):
        super().__init__()
        fb = librosa.filters.mel(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
        self.register_buffer("mel_fb", torch.tensor(fb, dtype=torch.float32))
        self.register_buffer("window", torch.hann_window(N_FFT))
        kernel = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0]) / 10.0   # regression delta, N = 2
        self.register_buffer("delta_kernel", kernel.view(1, 1, -1))

    def log_mel(self, wav):
        spec = torch.stft(wav, n_fft=N_FFT, hop_length=HOP_LENGTH, window=self.window,
                          center=True, return_complex=True)
        power = spec.abs() ** 2                       # (batch, freq bins, frames)
        mel = torch.matmul(self.mel_fb, power)        # energy in each mel band
        return torch.log(mel + 1e-6)[:, :, :N_FRAMES]

    def delta(self, x):
        b, m, t = x.shape
        padded = F.pad(x.reshape(b * m, 1, t), (2, 2), mode="replicate")
        return F.conv1d(padded, self.delta_kernel).reshape(b, m, t)

    def forward(self, wav):
        logmel = self.log_mel(wav)
        d1 = self.delta(logmel)
        d2 = self.delta(d1)
        x = torch.stack([logmel, d1, d2], dim=1)
        mean = x.mean(dim=(2, 3), keepdim=True)
        std = x.std(dim=(2, 3), keepdim=True)
        return (x - mean) / (std + 1e-5)


featurizer = LogMelDeltas().to(device).eval()
with torch.no_grad():
    INPUT_SHAPE = tuple(featurizer(torch.zeros(1, CROP_SAMPLES, device=device)).shape[1:])
print("Model input shape (channels, mel bands, frames):", INPUT_SHAPE)

In [ ]:
# Figure 3.2: Respiratory Distress compared with Non-distress (normal vocalization and environmental noise).
panels = [("Distress", "Respiratory Distress"),
          ("Normal", "Non-distress: normal vocalization"),
          ("Noise", "Non-distress: environmental noise")]
fig, axes = plt.subplots(len(panels), 2, figsize=(12, 7.5))
for r, (source, title) in enumerate(panels):
    idx = manifest.index[(manifest["role"] == "train/val/test") & (manifest["source"] == source)][0]
    wav = crop(clips[idx])
    with torch.no_grad():
        mel = featurizer.log_mel(torch.tensor(wav, device=device).unsqueeze(0))[0].cpu().numpy()
    axes[r, 0].plot(np.arange(len(wav)) / SAMPLE_RATE, wav, linewidth=0.5)
    axes[r, 0].set_title(f"{title} (waveform)")
    axes[r, 0].set_ylabel("Amplitude")
    im = axes[r, 1].imshow(mel, origin="lower", aspect="auto", cmap="magma",
                           extent=[0, len(wav) / SAMPLE_RATE, 0, N_MELS])
    axes[r, 1].set_title(f"{title} (log-mel spectrogram)")
    axes[r, 1].set_ylabel("Mel band")
    fig.colorbar(im, ax=axes[r, 1], label="log energy")
axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_3_2_distress_vs_non_distress.png"), dpi=200)
plt.show()

## 5. Splitting and augmentation (Sections 3.1.4 and 3.1.2)

**Splitting.** Labels are encoded as 0 = Non-distress and 1 = Distress. The data is divided into 70% training, 15% validation and 15% test, stratified by source category. Clips cut from the **same source recording always stay in the same split**, so the test set never contains another slice of a recording the model was trained on. Each seed produces a different split.

**Augmentation** is applied to the training split only, each time a clip is drawn:
- random position of the 4-second crop inside the clip (a time shift),
- random gain on the waveform,
- masking of a random block of time frames and of a random block of mel bands.

In [ ]:
def make_split(seed):
    """Returns {"train": indices, "val": indices, "test": indices} into `manifest`."""
    df = manifest.loc[main_idx]
    sgkf = StratifiedGroupKFold(n_splits=sum(SPLIT_FOLDS), shuffle=True, random_state=seed)
    fold_of = np.zeros(len(df), dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["source"], df[GROUP_BY])):
        fold_of[idx] = fold
    a, b = SPLIT_FOLDS[0], SPLIT_FOLDS[0] + SPLIT_FOLDS[1]
    split = {"train": main_idx[fold_of < a],
             "val": main_idx[(fold_of >= a) & (fold_of < b)],
             "test": main_idx[fold_of >= b]}
    groups = [set(manifest.loc[v, GROUP_BY]) for v in split.values()]
    assert not (groups[0] & groups[1]) and not (groups[0] & groups[2]) and not (groups[1] & groups[2])
    return split


def split_table(split):
    rows = {}
    for name, idx in split.items():
        src = manifest.loc[idx, "source"].value_counts()
        rows[name] = {"Normal": src.get("Normal", 0), "Noise": src.get("Noise", 0),
                      "Non-distress": src.get("Normal", 0) + src.get("Noise", 0),
                      "Distress": src.get("Distress", 0), "total": len(idx)}
    table = pd.DataFrame(rows).T
    table["share"] = (table["total"] / table["total"].sum()).round(3)
    return table


print(f"Split for seed {PRIMARY_SEED} (grouped by {GROUP_BY}):")
print(split_table(make_split(PRIMARY_SEED)).to_string())

In [ ]:
class ClipDataset(Dataset):
    """Returns (4-s waveform, binary label, row index in `manifest`)."""

    def __init__(self, indices, train=False, noise_pool=()):
        self.indices = np.asarray(indices)
        self.labels = manifest["label"].to_numpy()
        self.train = train
        self.noise_pool = list(noise_pool)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        clip = clips[idx]
        if not self.train:
            return torch.from_numpy(crop(clip)), int(self.labels[idx]), idx
        start = np.random.randint(0, max(1, len(clip) - CROP_SAMPLES + 1))   # random crop position
        x = crop(clip, start)
        if self.noise_pool and np.random.rand() < AUG_NOISE_PROB:            # optional noise mixing
            noise_clip = clips[self.noise_pool[np.random.randint(len(self.noise_pool))]]
            noise = crop(noise_clip, np.random.randint(0, max(1, len(noise_clip) - CROP_SAMPLES + 1)))
            snr_db = np.random.uniform(*AUG_SNR_DB)
            scale = np.sqrt((np.mean(x ** 2) + 1e-10) / ((np.mean(noise ** 2) + 1e-10) * 10 ** (snr_db / 10)))
            x = x + scale * noise
        x = x * 10.0 ** (np.random.uniform(*AUG_GAIN_DB) / 20.0)             # random gain
        return torch.from_numpy(np.clip(x, -1.0, 1.0).astype(np.float32)), int(self.labels[idx]), idx


def mask_features(x):
    """Time and frequency masking on a batch of feature tensors. Channels are standardized,
    so filling with 0 fills each channel with its own mean."""
    _, _, n_freq, n_time = x.shape
    for i in range(x.size(0)):
        if random.random() < AUG_TIME_MASK_PROB:
            width = random.randint(5, n_time // 8)
            t0 = random.randint(0, n_time - width)
            x[i, :, :, t0:t0 + width] = 0.0
        if random.random() < AUG_FREQ_MASK_PROB:
            height = random.randint(3, n_freq // 8)
            f0 = random.randint(0, n_freq - height)
            x[i, :, f0:f0 + height, :] = 0.0
    return x


def seed_worker(worker_id):
    seed = torch.initial_seed() % 2 ** 32
    np.random.seed(seed)
    random.seed(seed)


def make_loader(indices, train=False):
    noise_pool = [i for i in indices if manifest.at[i, "source"] == "Noise"] if train else ()
    return DataLoader(ClipDataset(indices, train, noise_pool), batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
                      pin_memory=(device.type == "cuda"))

## 6. Classification models (Section 3.2.1)

**CNN–BiLSTM (proposed).** Two convolutional blocks (convolution, batch normalization, ReLU, pooling) learn local time–frequency patterns, such as the broadband burst of a cough. Pooling turns the 4-second input into a sequence of 100 steps of about 40 ms each. The BiLSTM reads this sequence forward and backward and models how the local patterns are arranged across the 4 seconds, for example whether bursts repeat. The final forward state and the final backward state are joined, passed through dropout, and a fully connected layer outputs two logits.

**Comparison models. All are binary and use the same splits, seeds and training loop:**
- **Baseline CNN**: the same convolutional blocks with the BiLSTM removed. This is the ablation that shows what the BiLSTM adds.
- **DS-CNN**: a lightweight depthwise-separable CNN of the kind used for TinyML (a reimplementation, not the team's earlier 3-class model).
- **MobileNetV3-Small**: a published architecture, trained from scratch on the same data.
- **Majority class**: always predicts Non-distress. Added to the results table as the floor.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x))))


class CNNBiLSTM(nn.Module):
    def __init__(self, n_mels=N_MELS, cnn_channels=64, lstm_hidden=128, dropout=DROPOUT, num_classes=2):
        super().__init__()
        self.conv_blocks = nn.Sequential(ConvBlock(3, cnn_channels), ConvBlock(cnn_channels, cnn_channels))
        self.bilstm = nn.LSTM(input_size=cnn_channels * (n_mels // 4), hidden_size=lstm_hidden,
                              num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * lstm_hidden, num_classes)

    def forward(self, x):                                   # (batch, 3, mel, frames)
        x = self.conv_blocks(x)                             # (batch, C, mel/4, frames/4)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)      # one feature vector per time step
        _, (h_n, _) = self.bilstm(x)
        x = torch.cat([h_n[-2], h_n[-1]], dim=1)            # final forward + final backward state
        return self.fc(self.dropout(x))                     # two logits


class BaselineCNN(nn.Module):
    """The CNN–BiLSTM without its recurrent layer."""

    def __init__(self, n_mels=N_MELS, cnn_channels=64, dropout=DROPOUT, num_classes=2):
        super().__init__()
        self.conv_blocks = nn.Sequential(ConvBlock(3, cnn_channels), ConvBlock(cnn_channels, cnn_channels))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(cnn_channels * (n_mels // 4), num_classes)

    def forward(self, x):
        x = self.conv_blocks(x)
        x = x.mean(dim=3).flatten(1)                        # average over time
        return self.fc(self.dropout(x))


class DSBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.depthwise = nn.Conv2d(ch, ch, kernel_size=3, padding=1, groups=ch)
        self.bn1 = nn.BatchNorm2d(ch)
        self.pointwise = nn.Conv2d(ch, ch, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(ch)

    def forward(self, x):
        x = F.relu(self.bn1(self.depthwise(x)))
        return F.relu(self.bn2(self.pointwise(x)))


class DSCNN(nn.Module):
    def __init__(self, channels=128, n_blocks=4, dropout=DROPOUT, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, channels, kernel_size=3, stride=2, padding=1),
                                  nn.BatchNorm2d(channels), nn.ReLU())
        self.blocks = nn.Sequential(*[DSBlock(channels) for _ in range(n_blocks)])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        x = x.mean(dim=(2, 3))                              # global average pooling
        return self.fc(self.dropout(x))


def mobilenet_v3_small():
    return torchvision.models.mobilenet_v3_small(weights=None, num_classes=2)


def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


MODELS = {   # key: (display name, builder)
    "cnn_bilstm": ("CNN–BiLSTM (proposed)", CNNBiLSTM),
    "baseline_cnn": ("Baseline CNN (no BiLSTM)", BaselineCNN),
    "ds_cnn": ("DS-CNN", DSCNN),
    "mobilenet_v3_small": ("MobileNetV3-Small", mobilenet_v3_small),
}
PROPOSED = "cnn_bilstm"
for key, (name, builder) in MODELS.items():
    n = count_params(builder())
    print(f"{name:28s} {n:>10,} trainable parameters ({n * 4 / 1e6:.2f} MB as 32-bit floats)")

In [ ]:
# Where the CNN–BiLSTM parameters come from.
m = CNNBiLSTM()
d, h = m.bilstm.input_size, m.bilstm.hidden_size
breakdown = pd.DataFrame({
    "component": ["Convolutional blocks", "BiLSTM", "Fully connected layer"],
    "parameters": [count_params(m.conv_blocks), count_params(m.bilstm), count_params(m.fc)],
})
breakdown["share"] = (breakdown["parameters"] / breakdown["parameters"].sum()).map("{:.1%}".format)
print(breakdown.to_string(index=False))
print(f"\nBiLSTM input size d = {d} (64 channels x {N_MELS // 4} mel bins), hidden size h = {h}")
print(f"2 directions x 4 gates x (h*d + h*h + 2h) = {2 * 4 * (h * d + h * h + 2 * h):,}")
print(f"Total: {count_params(m):,}")

## 7. Training objective and optimization (Sections 3.2.2 and 3.2.3)

- **Softmax:** $\hat{y}_c = e^{z_c} / \sum_{j=1}^{C} e^{z_j}$, where $z_c$ is the logit for class $c$ and $C = 2$. It runs inside the loss during training and is applied explicitly at inference to get the Distress probability.
- **Class weights (Equation 3-1):** $w_c = \bar{N} / N_c$, computed from the training clips of each split.
- **Label smoothing (Equation 3-3):** $\tilde{y}_c = y_c(1-\varepsilon) + \varepsilon / C$ with $\varepsilon = 0.02$, so the targets are 0.99 and 0.01 instead of 1 and 0. $\varepsilon$ is a training setting, not an error rate.
- **Weighted cross-entropy (Equation 3-2):** $L = -\sum_{c} w_c\, \tilde{y}_c \log \hat{y}_c$ per sample, averaged over the batch.
- **Adam** with weight decay, a **scheduler** that halves the learning rate when the validation loss stops improving, and **early stopping** that keeps the weights from the epoch with the lowest validation loss.

The hyperparameters are the fixed values set in Part 1. They were not tuned by a search.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over a loader. Trains when an optimizer is given, otherwise evaluates."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for wav, y, _ in loader:
            wav, y = wav.to(device), y.to(device)
            with torch.no_grad():
                x = featurizer(wav)
                if training:
                    x = mask_features(x)
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


def train_model(key, seed, loaders, criterion):
    name, builder = MODELS[key]
    set_seed(seed)
    model = builder().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=1e-6)

    history, best_val_loss, best_state, stale = [], float("inf"), None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, loaders["train"], criterion, optimizer)
        val_loss, val_acc = run_epoch(model, loaders["val"], criterion)
        scheduler.step(val_loss)
        improved = val_loss < best_val_loss - 1e-4
        if improved:
            best_val_loss, stale = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())      # a real copy of the best weights
        else:
            stale += 1
        history.append({"epoch": epoch, "lr": optimizer.param_groups[0]["lr"],
                        "train_loss": train_loss, "train_acc": train_acc,
                        "val_loss": val_loss, "val_acc": val_acc, "best": improved})
        print(f"[{name} | seed {seed}] epoch {epoch:02d}/{MAX_EPOCHS} "
              f"| loss {train_loss:.4f} acc {train_acc:.4f} "
              f"| val_loss {val_loss:.4f} val_acc {val_acc:.4f} {'*' if improved else ''}")
        if stale >= EARLY_STOP_PATIENCE:
            print(f"[{name} | seed {seed}] early stopping at epoch {epoch}")
            break
    model.load_state_dict(best_state)
    return model.eval(), pd.DataFrame(history)


def predict(model, loader):
    """Distress probability (softmax) for every clip in a loader."""
    model.eval()
    probs, labels, rows = [], [], []
    with torch.no_grad():
        for wav, y, row in loader:
            logits = model(featurizer(wav.to(device)))
            probs.append(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
            labels.append(y.numpy())
            rows.append(row.numpy())
    return np.concatenate(probs), np.concatenate(labels), np.concatenate(rows)


def choose_threshold(y_val, p_val, target=TARGET_RECALL):
    """Highest threshold whose validation distress recall still reaches the target."""
    ok = [t for t in np.linspace(0.05, 0.95, 181)
          if recall_score(y_val, (p_val >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else DEFAULT_THRESHOLD


def metrics_at(y_true, prob, threshold):
    y_pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "distress_precision": precision_score(y_true, y_pred, zero_division=0),
            "distress_recall": recall_score(y_true, y_pred, zero_division=0),
            "distress_f1": f1_score(y_true, y_pred, zero_division=0),
            "fnr": fn / max(fn + tp, 1), "fpr": fp / max(fp + tn, 1)}

## 8. Training over several seeds

For every seed the data is split again and every model is trained once on that split. Each finished run is saved to `OUT_DIR/runs` (weights, predictions, history and metrics), so if Colab disconnects, running this cell again skips what is already done.

In [ ]:
def run_paths(key, seed):
    base = os.path.join(RUN_DIR, f"{key}_seed{seed}")
    return {"metrics": base + ".json", "weights": base + ".pt",
            "preds": base + "_predictions.npz", "history": base + "_history.csv"}


for seed in SEEDS:
    split = make_split(seed)
    pending = [k for k in MODELS if not os.path.exists(run_paths(k, seed)["metrics"])]
    if not pending:
        print(f"seed {seed}: all models already done")
        continue
    set_seed(seed)
    loaders = {"train": make_loader(split["train"], train=True),
               "val": make_loader(split["val"]), "test": make_loader(split["test"])}
    train_counts = np.bincount(manifest.loc[split["train"], "label"], minlength=2)
    class_weights = torch.tensor(train_counts.mean() / train_counts, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    print(f"\n=== seed {seed} | train clips per class {train_counts.tolist()} | "
          f"class weights {[round(w, 4) for w in class_weights.tolist()]} ===")

    for key in pending:
        t0 = time.time()
        model, history = train_model(key, seed, loaders, criterion)
        p_val, y_val, _ = predict(model, loaders["val"])
        p_test, y_test, rows_test = predict(model, loaders["test"])
        tuned = choose_threshold(y_val, p_val)
        result = {"model": key, "seed": seed, "parameters": count_params(model),
                  "epochs_run": len(history), "best_epoch": int(history.loc[history["best"], "epoch"].max()),
                  "train_minutes": (time.time() - t0) / 60, "tuned_threshold": tuned,
                  "class_weights": class_weights.tolist(), "roc_auc": roc_auc_score(y_test, p_test),
                  "at_0.5": metrics_at(y_test, p_test, DEFAULT_THRESHOLD),
                  "at_tuned": metrics_at(y_test, p_test, tuned)}
        paths = run_paths(key, seed)
        torch.save(model.state_dict(), paths["weights"])
        np.savez(paths["preds"], p_val=p_val, y_val=y_val, p_test=p_test, y_test=y_test, rows_test=rows_test)
        history.to_csv(paths["history"], index=False)
        with open(paths["metrics"], "w") as f:      # written last: marks the run as complete
            json.dump(result, f, indent=1)
        print(f"[{MODELS[key][0]} | seed {seed}] test accuracy {result['at_0.5']['accuracy']:.3f} | "
              f"distress recall {result['at_0.5']['distress_recall']:.3f} | ROC-AUC {result['roc_auc']:.3f} | "
              f"{result['train_minutes']:.1f} min\n")

In [ ]:
# Training and validation curves for the primary seed.
fig, axes = plt.subplots(2, len(MODELS), figsize=(4.6 * len(MODELS), 6.5), squeeze=False)
for col, (key, (name, _)) in enumerate(MODELS.items()):
    hist = pd.read_csv(run_paths(key, PRIMARY_SEED)["history"])
    best_epoch = int(hist.loc[hist["best"], "epoch"].max())
    for row, metric in enumerate(["loss", "acc"]):
        ax = axes[row, col]
        ax.plot(hist["epoch"], hist[f"train_{metric}"], label="train")
        ax.plot(hist["epoch"], hist[f"val_{metric}"], label="validation")
        ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1, label="best checkpoint")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss" if metric == "loss" else "Accuracy")
        ax.set_title(name if row == 0 else "")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=200)
plt.show()

## 9. Evaluation on the held-out test set (Sections 3.2.4 and 3.3.5)

Every model is evaluated on the test set of its split, which was not used for training, model selection or threshold selection. Metrics are computed per clip with Distress as the positive class, and reported as **mean ± standard deviation over the seeds**.

Two operating points are reported:
- **Threshold 0.5** on the Distress probability.
- **Recall-target threshold**: the highest threshold that still reaches the target distress recall on the *validation* set. Missing distress is the costly error, so the threshold is set from recall, not from accuracy.

The reference points are the **majority-class baseline**, the **baseline CNN**, and **MobileNetV3-Small**. The Proportional Chance Criterion is printed only as a minimum check.

In [ ]:
def inference_ms(model, repeats=30):
    """Mean CPU time to classify one 4-s clip: (feature extraction ms, model ms)."""
    cpu_model = copy.deepcopy(model).cpu().eval()
    cpu_feat = copy.deepcopy(featurizer).cpu().eval()
    wav = torch.randn(1, CROP_SAMPLES)
    with torch.no_grad():
        x = cpu_feat(wav)
        for _ in range(5):
            cpu_model(x)
        t0 = time.perf_counter()
        for _ in range(repeats):
            cpu_feat(wav)
        feat_ms = (time.perf_counter() - t0) / repeats * 1000.0
        t0 = time.perf_counter()
        for _ in range(repeats):
            cpu_model(x)
        model_ms = (time.perf_counter() - t0) / repeats * 1000.0
    return feat_ms, model_ms


def load_run(key, seed):
    paths = run_paths(key, seed)
    with open(paths["metrics"]) as f:
        return json.load(f), np.load(paths["preds"])


def load_model(key, seed):
    model = MODELS[key][1]()
    model.load_state_dict(torch.load(run_paths(key, seed)["weights"], map_location="cpu"))
    return model.to(device).eval()


rows = []
for seed in SEEDS:
    for key in MODELS:
        r, preds = load_run(key, seed)
        rows.append({"model": MODELS[key][0], "seed": seed, "parameters": r["parameters"],
                     "epochs_run": r["epochs_run"], "roc_auc": r["roc_auc"],
                     "tuned_threshold": r["tuned_threshold"],
                     **{f"{k}": v for k, v in r["at_0.5"].items()},
                     **{f"tuned_{k}": v for k, v in r["at_tuned"].items()}})
    # Majority class: always predict Non-distress.
    y = preds["y_test"]
    rows.append({"model": "Majority class", "seed": seed, "parameters": 0, "epochs_run": 0, "roc_auc": 0.5,
                 "tuned_threshold": np.nan, **metrics_at(y, np.zeros(len(y)), DEFAULT_THRESHOLD),
                 **{f"tuned_{k}": v for k, v in metrics_at(y, np.zeros(len(y)), DEFAULT_THRESHOLD).items()}})
all_runs = pd.DataFrame(rows)
all_runs.to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)

order = ["Majority class"] + [MODELS[k][0] for k in MODELS if k != PROPOSED] + [MODELS[PROPOSED][0]]


def mean_sd(df, cols):
    g = df.groupby("model")[cols]
    out = g.mean().round(3).astype(str) + " ± " + g.std(ddof=0).round(3).astype(str)
    return out.reindex(order)


latency = {MODELS[k][0]: inference_ms(load_model(k, PRIMARY_SEED)) for k in MODELS}
info = all_runs.groupby("model")[["parameters"]].first().reindex(order)
info["size_MB"] = (info["parameters"] * 4 / 1e6).round(2)
info["CPU_ms_model"] = [round(latency[m][1], 1) if m in latency else 0.0 for m in order]

table_3_1 = info.join(mean_sd(all_runs, ["accuracy", "macro_f1", "distress_recall", "distress_f1", "roc_auc"]))
table_3_1.to_csv(os.path.join(OUT_DIR, "table_3_1_model_comparison.csv"))
print(f"Table 3.1 Model comparison at threshold 0.5 (mean ± SD over {len(SEEDS)} seeds)")
print(table_3_1.to_string())
feat_ms = next(iter(latency.values()))[0]
print(f"\nFeature extraction adds {feat_ms:.1f} ms per clip. Times are from this Colab CPU, not the Raspberry Pi 4.")

tuned_table = mean_sd(all_runs[all_runs["model"] != "Majority class"],
                      ["tuned_threshold", "tuned_distress_recall", "tuned_distress_precision",
                       "tuned_fnr", "tuned_fpr", "tuned_accuracy"]).dropna()
tuned_table.to_csv(os.path.join(OUT_DIR, "table_recall_target_threshold.csv"))
print(f"\nAt the recall-target threshold (validation distress recall >= {TARGET_RECALL}):")
print(tuned_table.to_string())

In [ ]:
# Does the BiLSTM help? Same seeds and splits, so the runs can be compared in pairs.
paired = all_runs.pivot(index="seed", columns="model", values=["accuracy", "distress_f1", "roc_auc"])
prop, base = MODELS[PROPOSED][0], MODELS["baseline_cnn"][0]
diff = pd.DataFrame({m: paired[m][prop] - paired[m][base] for m in ["accuracy", "distress_f1", "roc_auc"]})
print(f"{prop} minus {base}, per seed:")
print(diff.round(4).to_string())
print("\nMean difference:", diff.mean().round(4).to_dict())
print("Seeds where the CNN–BiLSTM is ahead:", (diff > 0).sum().to_dict(), f"out of {len(SEEDS)}")
diff.to_csv(os.path.join(OUT_DIR, "paired_cnn_bilstm_vs_baseline.csv"))

In [ ]:
# Detailed results for the proposed model on the primary seed.
result, preds = load_run(PROPOSED, PRIMARY_SEED)
y_test, p_test, rows_test = preds["y_test"], preds["p_test"], preds["rows_test"]
TUNED_THRESHOLD = result["tuned_threshold"]
pred_05 = (p_test >= DEFAULT_THRESHOLD).astype(int)
print(f"{MODELS[PROPOSED][0]}, seed {PRIMARY_SEED}, threshold {DEFAULT_THRESHOLD}")
print(classification_report(y_test, pred_05, target_names=CLASS_NAMES, digits=3))
print(f"Recall-target threshold for this run: {TUNED_THRESHOLD:.3f} ->",
      {k: round(v, 3) for k, v in result["at_tuned"].items()})

# Proportional Chance Criterion (Equations 3-12 and 3-13): a minimum check only.
q = y_test.mean()
p = 1.0 - q
pcc = p ** 2 + q ** 2
test_acc = accuracy_score(y_test, pred_05)
print(f"\nTest proportions: Non-distress p = {p:.3f}, Distress q = {q:.3f}")
print(f"PCC = p² + q² = {pcc:.3f} | 1.25 × PCC = {1.25 * pcc:.3f} | majority-class accuracy = {max(p, q):.3f}")
print(f"Test accuracy {test_acc:.3f} -> {'exceeds' if test_acc > 1.25 * pcc else 'does NOT exceed'} 1.25 × PCC")

In [ ]:
# Confusion matrices (threshold 0.5) and ROC curves, primary seed.
fig, axes = plt.subplots(1, len(MODELS) + 1, figsize=(4.4 * (len(MODELS) + 1), 4.2))
for ax, (key, (name, _)) in zip(axes, MODELS.items()):
    _, pr = load_run(key, PRIMARY_SEED)
    cm = confusion_matrix(pr["y_test"], (pr["p_test"] >= DEFAULT_THRESHOLD).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(CLASS_NAMES, rotation=90, va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(name, fontsize=10)
    fpr, tpr, _ = roc_curve(pr["y_test"], pr["p_test"])
    axes[-1].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(pr['y_test'], pr['p_test']):.3f})")
axes[-1].plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
axes[-1].set_xlabel("False positive rate")
axes[-1].set_ylabel("True positive rate")
axes[-1].set_title("ROC curves")
axes[-1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrices_and_roc.png"), dpi=200)
plt.show()

In [ ]:
# Non-distress errors examined separately for normal and noise clips, and a predictions file
# that links every test clip back to its audio file (Section 3.3.5).
test_predictions = manifest.loc[rows_test, ["path", "dataset", "recording", "original_label", "source"]].copy()
test_predictions["true_label"] = y_test
test_predictions["prob_distress"] = p_test
test_predictions["pred_at_0.5"] = pred_05
test_predictions["pred_at_tuned"] = (p_test >= TUNED_THRESHOLD).astype(int)

by_source = test_predictions.groupby("source").agg(
    clips=("true_label", "size"),
    flagged_at_05=("pred_at_0.5", "mean"), flagged_at_tuned=("pred_at_tuned", "mean"))
print("Share of clips classified as Distress, by source category (Normal and Noise are both Non-distress):")
print(by_source.round(3).to_string())

pred_path = os.path.join(OUT_DIR, "test_predictions_cnn_bilstm.csv")
test_predictions.to_csv(pred_path, index=False)

# Model file for the gateway, with everything needed to repeat the preprocessing.
proposed_model = load_model(PROPOSED, PRIMARY_SEED)
torch.save({"state_dict": proposed_model.state_dict(), "class_names": CLASS_NAMES,
            "threshold_default": DEFAULT_THRESHOLD, "threshold_recall_target": TUNED_THRESHOLD,
            "preprocessing": {"sample_rate": SAMPLE_RATE, "crop_sec": CROP_SEC, "n_fft": N_FFT,
                              "hop_length": HOP_LENGTH, "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX}},
           os.path.join(OUT_DIR, "sonara_cnn_bilstm.pt"))
print("\nSaved:", pred_path, "and sonara_cnn_bilstm.pt")

## 10. External test on PVSD

PVSD is never used for training or for choosing the threshold. Each PVSD file is cut into consecutive 4-second crops (short files are repeated to 4 seconds), the Distress probabilities of the crops are averaged, and the file gets one prediction. This shows how a model trained on SmartEars behaves on recordings from a different source.

This part is skipped if there is no `PVSD` folder in the repo.

In [ ]:
def predict_file(model, clip):
    """Mean Distress probability over consecutive 4-s crops of one file."""
    starts = list(range(0, max(1, len(clip) - CROP_SAMPLES + 1), CROP_SAMPLES))
    batch = torch.from_numpy(np.stack([crop(clip, s) for s in starts])).to(device)
    with torch.no_grad():
        return F.softmax(model(featurizer(batch)), dim=1)[:, 1].mean().item()


if len(external_idx) == 0:
    print("No external dataset found. Add a PVSD folder (Healthy / Unhealthy / Noise) to the repo to run this part.")
else:
    y_ext = manifest.loc[external_idx, "label"].to_numpy()
    ext_rows = []
    for seed in SEEDS:
        for key in MODELS:
            model, (r, _) = load_model(key, seed), load_run(key, seed)
            p_ext = np.array([predict_file(model, clips[i]) for i in external_idx])
            ext_rows.append({"model": MODELS[key][0], "seed": seed, "roc_auc": roc_auc_score(y_ext, p_ext),
                             **metrics_at(y_ext, p_ext, DEFAULT_THRESHOLD)})
            if key == PROPOSED and seed == PRIMARY_SEED:
                ext_predictions = manifest.loc[external_idx, ["path", "dataset", "original_label", "source"]].copy()
                ext_predictions["true_label"], ext_predictions["prob_distress"] = y_ext, p_ext
                ext_predictions["pred_at_0.5"] = (p_ext >= DEFAULT_THRESHOLD).astype(int)
                ext_predictions.to_csv(os.path.join(OUT_DIR, "external_predictions_cnn_bilstm.csv"), index=False)
    external_runs = pd.DataFrame(ext_rows)
    external_runs.to_csv(os.path.join(OUT_DIR, "external_runs.csv"), index=False)
    g = external_runs.groupby("model")[["accuracy", "macro_f1", "distress_recall", "distress_f1", "roc_auc"]]
    external_table = (g.mean().round(3).astype(str) + " ± " + g.std(ddof=0).round(3).astype(str))
    print(f"External test: {len(external_idx)} files, Non-distress {int((y_ext == 0).sum())} / Distress {int((y_ext == 1).sum())}")
    print(external_table.reindex([m for m in order if m in external_table.index]).to_string())
    print("\nCNN–BiLSTM, share of files classified as Distress by original label:")
    print(ext_predictions.groupby("original_label")["pred_at_0.5"].agg(["size", "mean"]).round(3).to_string())

## 11. Gateway inference and zone-level attribution (Sections 3.3.2 to 3.3.4)

This part implements the gateway logic: parse a packet, preprocess the audio the same way as in training, classify it, apply the decision threshold, look up the zone of the sending node, log the event, and raise a zone alert when the share of Distress detections in a moving window passes a threshold.

**This is a software simulation.** Held-out test clips stand in for audio packets, and the node assignment is made up. It checks that the gateway logic works and shows how clip-level errors carry over to zone-level alerts. It is **not** the controlled playback test or the field comparison with veterinary observation, which need the physical nodes. It also says nothing about whether 4 seconds of audio can be sent over the LoRa link in time; that depends on the link decision.

In [ ]:
NODE_TO_ZONE = {"node-01": "Zone 1", "node-02": "Zone 2", "node-03": "Zone 3"}
ALERT_WINDOW = 20         # number of most recent clips considered per zone
ALERT_RATE = 0.40         # alert when this share of them is classified as Distress
PACKET_INTERVAL_MIN = 0.5
GATEWAY_THRESHOLD = TUNED_THRESHOLD


class Gateway:
    def __init__(self, model, node_to_zone, threshold):
        self.model = model.eval()
        self.node_to_zone = node_to_zone
        self.threshold = threshold
        self.recent = {zone: deque(maxlen=ALERT_WINDOW) for zone in node_to_zone.values()}
        self.alert_active = {zone: False for zone in node_to_zone.values()}
        self.event_log, self.alerts = [], []

    def process_packet(self, packet):
        # 1. Parse the packet.
        node_id, audio, sr, t_min = packet["node_id"], packet["audio"], packet["sample_rate"], packet["t_min"]
        # 2. Same preprocessing as training: mono, 16 kHz, 4-s crop.
        audio = np.asarray(audio, dtype=np.float32)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != SAMPLE_RATE:
            g = gcd(int(sr), SAMPLE_RATE)
            audio = resample_poly(audio, SAMPLE_RATE // g, sr // g).astype(np.float32)
        audio = crop(np.clip(audio * 32767.0, -32768, 32767).astype(np.int16))
        # 3-4. Three-channel tensor -> CNN–BiLSTM -> class probabilities.
        with torch.no_grad():
            x = featurizer(torch.from_numpy(audio).unsqueeze(0).to(device))
            prob = F.softmax(self.model(x), dim=1)[0, 1].item()
        # 5. Decision threshold on the Distress probability.
        pred = int(prob >= self.threshold)
        # Zone attribution: fixed node-to-zone lookup (Section 3.3.3).
        zone = self.node_to_zone[node_id]
        # Zone-level aggregation and alert.
        self.recent[zone].append(pred)
        rate = sum(self.recent[zone]) / len(self.recent[zone])
        full = len(self.recent[zone]) == ALERT_WINDOW
        if full and rate >= ALERT_RATE and not self.alert_active[zone]:
            self.alert_active[zone] = True
            self.alerts.append({"t_min": t_min, "zone": zone, "distress_rate": rate})
        elif full and rate < ALERT_RATE:
            self.alert_active[zone] = False
        event = {"t_min": t_min, "node_id": node_id, "zone": zone, "predicted_class": CLASS_NAMES[pred],
                 "prob_distress": round(prob, 4), "zone_distress_rate": round(rate, 3)}
        self.event_log.append(event)
        return event

In [ ]:
# Simulated scenario: three zones. Zones 1 and 3 stay healthy. Zone 2 is healthy for the
# first half, then 60% of its clips contain respiratory distress.
rng = np.random.default_rng(PRIMARY_SEED)
pool = {0: rows_test[y_test == 0], 1: rows_test[y_test == 1]}
PACKETS_PER_ZONE = 120
OUTBREAK_ZONE, OUTBREAK_START, OUTBREAK_SHARE = "Zone 2", 60, 0.60

gateway = Gateway(proposed_model, NODE_TO_ZONE, GATEWAY_THRESHOLD)
truth = []
for k in range(PACKETS_PER_ZONE):
    for node_id, zone in NODE_TO_ZONE.items():
        distress = int(zone == OUTBREAK_ZONE and k >= OUTBREAK_START and rng.random() < OUTBREAK_SHARE)
        clip = clips[int(rng.choice(pool[distress]))]
        gateway.process_packet({"node_id": node_id, "audio": clip.astype(np.float32) / 32768.0,
                                "sample_rate": SAMPLE_RATE, "t_min": k * PACKET_INTERVAL_MIN})
        truth.append(distress)

event_log = pd.DataFrame(gateway.event_log)
event_log["true_class"] = [CLASS_NAMES[t] for t in truth]
event_log.to_csv(os.path.join(OUT_DIR, "simulated_gateway_event_log.csv"), index=False)
print(f"Gateway threshold: {GATEWAY_THRESHOLD:.3f}. Event log (first rows):")
print(event_log.head(6).to_string(index=False))

In [ ]:
is_true = event_log["true_class"] == "Distress"
is_pred = event_log["predicted_class"] == "Distress"
zone_summary = pd.DataFrame({
    "clips": event_log.groupby("zone").size(),
    "true Distress": is_true.groupby(event_log["zone"]).sum(),
    "detected Distress": is_pred.groupby(event_log["zone"]).sum(),
    "false alarms": (is_pred & ~is_true).groupby(event_log["zone"]).sum(),
    "missed": (~is_pred & is_true).groupby(event_log["zone"]).sum(),
})
alerts = pd.DataFrame(gateway.alerts, columns=["t_min", "zone", "distress_rate"])
zone_summary["alerts raised"] = alerts.groupby("zone").size().reindex(zone_summary.index).fillna(0).astype(int)
zone_summary["first alert (min)"] = alerts.groupby("zone")["t_min"].min().reindex(zone_summary.index)
print(f"Simulated outbreak: {OUTBREAK_ZONE} from minute {OUTBREAK_START * PACKET_INTERVAL_MIN:.0f}")
print(zone_summary.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
for zone, g in event_log.groupby("zone"):
    ax.plot(g["t_min"], g["zone_distress_rate"], label=zone)
ax.axhline(ALERT_RATE, color="red", linestyle="--", linewidth=1, label="alert threshold")
ax.axvline(OUTBREAK_START * PACKET_INTERVAL_MIN, color="gray", linestyle=":", label=f"outbreak starts ({OUTBREAK_ZONE})")
ax.set_xlabel("Simulated time (minutes)")
ax.set_ylabel(f"Share of last {ALERT_WINDOW} clips classified as Distress")
ax.set_title("Zone-level distress rate (simulation)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "simulated_zone_distress_rate.png"), dpi=200)
plt.show()

## 12. Values to copy into the manuscript

The cell below prints the facts and numbers that Sections 3.1 and 3.2 need. All figures, tables and CSV files are in `OUT_DIR`.

In [ ]:
split = make_split(PRIMARY_SEED)
print("Section 3.1")
print(f"  Training datasets: {', '.join(TRAIN_DATASETS)} ({len(main_idx):,} clips). "
      f"External test: {len(external_idx):,} files.")
print(f"  Input: {SAMPLE_RATE} Hz mono, one {CROP_SEC:.0f}-s crop per file, {N_MELS} log-mel bands "
      f"({N_FFT / SAMPLE_RATE * 1000:.0f} ms window, {HOP_LENGTH / SAMPLE_RATE * 1000:.0f} ms hop) "
      f"plus delta and delta-delta -> {INPUT_SHAPE[0]} x {INPUT_SHAPE[1]} x {INPUT_SHAPE[2]}.")
print(f"  Split: 70/15/15 by {GROUP_BY}, stratified by source category; clips per split (seed {PRIMARY_SEED}): "
      f"{ {k: len(v) for k, v in split.items()} }")
print("Section 3.2.1")
print(f"  Hyperparameters were set by hand and not tuned: Adam, learning rate {LEARNING_RATE}, "
      f"weight decay {WEIGHT_DECAY}, batch size {BATCH_SIZE}, dropout {DROPOUT}.")
print(f"  Maximum {MAX_EPOCHS} epochs; early stopping patience {EARLY_STOP_PATIENCE}; {len(SEEDS)} seeds: {SEEDS}.")
epochs = all_runs[all_runs["model"] != "Majority class"].groupby("model")["epochs_run"].agg(["min", "max"])
for model_name, r in epochs.iterrows():
    feat, mod = latency[model_name]
    print(f"  {model_name}: stopped after {r['min']} to {r['max']} epochs; {mod:.1f} ms per clip "
          f"(+ {feat:.1f} ms features) on the Colab CPU. Measure again on the Raspberry Pi 4.")
print("Section 3.2.2")
print(f"  Label smoothing factor {LABEL_SMOOTHING}, set by hand (not selected on the validation set).")
print("Section 3.3.2")
print(f"  Decision threshold: results at {DEFAULT_THRESHOLD} and at a validation recall target of {TARGET_RECALL}.")